# Pipeline Playbook

Notebook runner for the full `free-exercise-graph` pipeline.

Safe defaults:
- backs up SQLite before work
- does a deterministic rebuild by default
- does **not** call the LLM unless you opt in
- does **not** do a destructive reset unless you opt in
- rebuilds the Phase 1 similarity graph and static app payloads after the graph is current

For the prose runbook, see `docs/full_run_playbook.md`.


In [1]:
from pathlib import Path
import os
import shlex
import subprocess
import sys

PROJECT_ROOT = Path.cwd()
assert (PROJECT_ROOT / "pipeline").exists(), "Run this notebook from the project root"
PYTHON = shlex.quote(sys.executable)

def run(cmd: str, *, check: bool = True) -> subprocess.CompletedProcess:
    print(f"$ {cmd}")
    result = subprocess.run(
        cmd,
        shell=True,
        cwd=PROJECT_ROOT,
        text=True,
        capture_output=True,
    )
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {cmd}")
    return result

print({'project_root': str(PROJECT_ROOT), 'python': sys.executable})

{'project_root': '/Users/talha/Code/free-exercise-graph', 'python': '/usr/local/bin/python3.11'}


## Environment Setup

Run this cell once in a new kernel. It installs the project into the same Python environment the notebook is using.

In [2]:
run(f"{PYTHON} -m pip install -e .")

$ /usr/local/bin/python3.11 -m pip install -e .
Obtaining file:///Users/talha/Code/free-exercise-graph
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'

ERROR: Exception:
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/pip/_internal/cli/base_command.py", line 107, in _run_wrapper
    status = _inner_run()
             ^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/pip/_internal/cli/base_command.py", line 98, in _inner_run
    return self.run(options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/pip/_internal/cli/req_command.py", line 96, in wrapper
    return func(self, options, args)


RuntimeError: Command failed with exit code 2: /usr/local/bin/python3.11 -m pip install -e .

## Configuration

Edit these flags before running the workflow cells.

In [3]:
PROVIDER = "anthropic"  # or "gemini"
MODEL = "claude-sonnet-4-6"             # e.g. "claude-sonnet-4-6" or "gemini-3.1-pro-preview"
CONCURRENCY = 3

DO_ENRICH = True
DO_DESTRUCTIVE_RESET = True
DO_IMPORT_EXISTING_EXPORT = False
DO_BUILD_SIMILARITY_GRAPH = True
DO_BUILD_APP = True

EXPORT_PATH_TO_IMPORT = None  # e.g. "pipeline/exports/enrichment-20260324-120000.jsonl"
SIMILARITY_GRAPH_INPUT = "graph.ttl"
SIMILARITY_GRAPH_OUT = "data/generated"
APP_OUT = "app"

print({
    "PROVIDER": PROVIDER,
    "MODEL": MODEL,
    "CONCURRENCY": CONCURRENCY,
    "DO_ENRICH": DO_ENRICH,
    "DO_DESTRUCTIVE_RESET": DO_DESTRUCTIVE_RESET,
    "DO_IMPORT_EXISTING_EXPORT": DO_IMPORT_EXISTING_EXPORT,
    "DO_BUILD_SIMILARITY_GRAPH": DO_BUILD_SIMILARITY_GRAPH,
    "DO_BUILD_APP": DO_BUILD_APP,
    "EXPORT_PATH_TO_IMPORT": EXPORT_PATH_TO_IMPORT,
    "SIMILARITY_GRAPH_INPUT": SIMILARITY_GRAPH_INPUT,
    "SIMILARITY_GRAPH_OUT": SIMILARITY_GRAPH_OUT,
    "APP_OUT": APP_OUT,
})


{'PROVIDER': 'anthropic', 'MODEL': 'claude-sonnet-4-6', 'CONCURRENCY': 3, 'DO_ENRICH': True, 'DO_DESTRUCTIVE_RESET': True, 'DO_IMPORT_EXISTING_EXPORT': False, 'EXPORT_PATH_TO_IMPORT': None}


## 1. Snapshot Current Local State

Run this before any risky work. The export may legitimately contain `0` entities if the current DB has no persisted enrichment state.

In [4]:
run(f"{PYTHON} pipeline/db_backup.py backup")
run(f"{PYTHON} pipeline/export_enrichment.py", check=False)

$ /usr/local/bin/python3.11 pipeline/db_backup.py backup
Created backup: /Users/talha/Code/free-exercise-graph/pipeline/backups/pipeline-20260324-134445.db

$ /usr/local/bin/python3.11 pipeline/export_enrichment.py
Exported 0 enriched entities to /Users/talha/Code/free-exercise-graph/pipeline/exports/enrichment-20260324-174445.jsonl



CompletedProcess(args='/usr/local/bin/python3.11 pipeline/export_enrichment.py', returncode=0, stdout='Exported 0 enriched entities to /Users/talha/Code/free-exercise-graph/pipeline/exports/enrichment-20260324-174445.jsonl\n', stderr='')

## 2. Optional Destructive Reset

Leave `DO_DESTRUCTIVE_RESET = False` unless you intentionally want to rebuild SQLite from scratch.

In [6]:
if DO_DESTRUCTIVE_RESET:
    run(f"{PYTHON} pipeline/run.py --reset-db --yes-reset-db --to build")
else:
    print("Skipping destructive reset. Set DO_DESTRUCTIVE_RESET = True to enable it.")

$ /usr/local/bin/python3.11 pipeline/run.py --reset-db --yes-reset-db --to build
Automatic backup before reset: /Users/talha/Code/free-exercise-graph/pipeline/backups/pipeline-20260324-134510.db
Resetting pipeline database...

== canonicalize ==
Loading free-exercise-db...
  873 exercises
  Wrote 873 records, 796 claims
Loading functional-fitness-db...
  3242 exercises
  Wrote 3242 records, 31369 claims

== identity ==
Entities:  3455
Merged:    577
Triage:    61

== reconcile ==
Resolved claims: 27256
Conflicts:       8
Deferred:        8

== build ==
Loading vocabulary (10 files)...
Assembling 3455 entities (0 enriched)...
Wrote /Users/talha/Code/free-exercise-graph/graph.ttl (58669 triples)




## 3. Deterministic Rebuild

This is the safe default pipeline path: `canonicalize -> identity -> reconcile -> build`.

In [7]:
run(f"{PYTHON} pipeline/run.py --to build")

$ /usr/local/bin/python3.11 pipeline/run.py --to build

== canonicalize ==
Loading free-exercise-db...
  873 exercises
  Wrote 873 records, 796 claims
Loading functional-fitness-db...
  3242 exercises
  Wrote 3242 records, 31369 claims

== identity ==
Entities:  3455
Merged:    577
Triage:    61

== reconcile ==
Resolved claims: 27256
Conflicts:       8
Deferred:        8

== build ==
Loading vocabulary (10 files)...
Assembling 3455 entities (0 enriched)...
Wrote /Users/talha/Code/free-exercise-graph/graph.ttl (58669 triples)




CompletedProcess(args='/usr/local/bin/python3.11 pipeline/run.py --to build', returncode=0, stdout='\n== canonicalize ==\nLoading free-exercise-db...\n  873 exercises\n  Wrote 873 records, 796 claims\nLoading functional-fitness-db...\n  3242 exercises\n  Wrote 3242 records, 31369 claims\n\n== identity ==\nEntities:  3455\nMerged:    577\nTriage:    61\n\n== reconcile ==\nResolved claims: 27256\nConflicts:       8\nDeferred:        8\n\n== build ==\nLoading vocabulary (10 files)...\nAssembling 3455 entities (0 enriched)...\nWrote /Users/talha/Code/free-exercise-graph/graph.ttl (58669 triples)\n', stderr='WARNING: building a deterministic-only graph (0 enriched entities). If you expected LLM-enriched output, restore/import enrichment state before shipping this graph.\n')

## 4. Optional Restore of an Existing Enrichment Export

Use this if you rebuilt deterministic state and want to restore a prior JSONL export.

In [8]:
if DO_IMPORT_EXISTING_EXPORT:
    assert EXPORT_PATH_TO_IMPORT, "Set EXPORT_PATH_TO_IMPORT first"
    run(f"{PYTHON} pipeline/import_enrichment.py {shlex.quote(EXPORT_PATH_TO_IMPORT)}")
    run(f"{PYTHON} pipeline/build.py")
else:
    print("Skipping import. Set DO_IMPORT_EXISTING_EXPORT = True and EXPORT_PATH_TO_IMPORT to use this step.")

Skipping import. Set DO_IMPORT_EXISTING_EXPORT = True and EXPORT_PATH_TO_IMPORT to use this step.


## 5. Inspect Pending Enrichment

This does not spend tokens.

In [9]:
run(f"{PYTHON} pipeline/enrich.py --dry-run")

$ /usr/local/bin/python3.11 pipeline/enrich.py --dry-run
Pending:     3455 / 3455 entities
Done:        0
Quarantined: 0 (≥3 failures)



CompletedProcess(args='/usr/local/bin/python3.11 pipeline/enrich.py --dry-run', returncode=0, stdout='Pending:     3455 / 3455 entities\nDone:        0\nQuarantined: 0 (≥3 failures)\n', stderr='')

## 6. Optional Full Enrichment Run

When enabled, the runner will:
- call the LLM
- archive raw responses under `pipeline/artifacts/raw_responses/`
- auto-export JSONL under `pipeline/exports/`
- rebuild `graph.ttl`


In [10]:
if DO_ENRICH:
    cmd = f"{PYTHON} pipeline/run.py --with-enrich --to build --concurrency {CONCURRENCY}"
    if PROVIDER:
        cmd += f" --provider {shlex.quote(PROVIDER)}"
    if MODEL:
        cmd += f" --model {shlex.quote(MODEL)}"
    run(cmd)
else:
    print("Skipping paid enrichment. Set DO_ENRICH = True to enable it.")

$ /usr/local/bin/python3.11 pipeline/run.py --with-enrich --to build --concurrency 3 --provider anthropic --model claude-sonnet-4-6

== canonicalize ==
Loading free-exercise-db...
  873 exercises
  Wrote 873 records, 796 claims
Loading functional-fitness-db...
  3242 exercises
  Wrote 3242 records, 31369 claims

== identity ==
Entities:  3455
Merged:    577
Triage:    61

== reconcile ==
Resolved claims: 27256
Conflicts:       8
Deferred:        8

== enrich ==
Loading ontology...
Enriching 3455 entities (provider=AnthropicProvider model=claude-sonnet-4-6 concurrency=3)...
Archiving raw enrichment outputs to /Users/talha/Code/free-exercise-graph/pipeline/artifacts/raw_responses/20260324-174730-enrich-anthropicprovider-claude-sonnet-4-6
  ✅ [1/3455] Alternating Double Dumbbell Bench Press  in=155 out=338
  ✅ [2/3455] Bodyweight Alternating Cossack Squat  in=159 out=484
  ✅ [3/3455] Bodyweight Alternating Curtsy Lunge  in=164 out=517
  ✅ [4/3455] Alternating Double Dumbbell Bicep Curl  i

## 7. Quality Surfaces

In [ ]:
run(f"{PYTHON} pipeline/validate.py --verbose")
run(f"{PYTHON} test_shacl.py") # honestly idk why this runs at the end

$ /usr/local/bin/python3.11 pipeline/validate.py --verbose

Data Quality Scorecard — free-exercise-graph
────────────────────────────────────────────────────────────
  Dimension       Status  Summary
────────────────────────────────────────────────────────────
  Validity        ✓ PASS   Skipped (use --shacl to run; slow on full graph). Shape unit tests: test_shacl.py
  Uniqueness      ✓ PASS   0 duplicate exercise labels
  Integrity       ✓ PASS   All vocab references resolve
  Timeliness      ⚠ WARN   14 unresolved warning(s) across 9 term(s): 'AntiRotation' (1 exercise), 'HipHinge' (5 exercises), 'HipRotation' (1 exercise), 'HipShift' (1 exercise), 'InfraspinatusHead' (2 exercises), 'Piriformis' (1 exercise), 'RearDelt' (1 exercise), 'SerrratusAnterior' (1 exercise), 'TrapezTrapezius' (1 exercise)
  Completeness    ⚠ WARN   506/3,455 exercises missing movementPattern
────────────────────────────────────────────────────────────

  [Timeliness]
  'AntiRotation': 1 exercise(s) not yet r

CompletedProcess(args='/usr/local/bin/python3.11 test_shacl.py', returncode=0, stdout='Running SHACL constraint tests\n\n  PASS  T00 baseline valid exercise (no violations)\n  PASS  T01 no movementPattern (optional since ADR-059) (no violations)\n  PASS  T02 no hasInvolvement\n  PASS  T03 no PrimeMover or PassiveTarget\n  PASS  T04 duplicate muscle\n  PASS  T05 ancestor+child double-counting\n  PASS  T06 Core as PrimeMover\n  PASS  T07 invalid involvement degree\n  PASS  T08 useGroupLevel head (RhomboidMajor)\n  PASS  T09 isCompound multiple values\n  PASS  T10 primaryJointAction using grouping node\n  PASS  T11 no primaryJointAction (non-passive)\n  PASS  T11b no primaryJointAction (passive exercise — exempt) (no violations)\n  PASS  T12 JA in both primary and supporting\n\n──────────────────────────────────────────────────\nResults: 14 passed  |  0 failed\n', stderr='')

## 8. Build Similarity Graph + Static App Payload

Phase 1 adds an offline similarity-graph build step after `graph.ttl` is current. The static app then consumes the generated neighbors and communities rather than computing substitutes in the browser.


In [ ]:
if DO_BUILD_SIMILARITY_GRAPH:
    run(
        f"{PYTHON} scripts/build_similarity_graph.py --input {shlex.quote(SIMILARITY_GRAPH_INPUT)} --out {shlex.quote(SIMILARITY_GRAPH_OUT)}"
    )
else:
    print("Skipping similarity graph build. Set DO_BUILD_SIMILARITY_GRAPH = True to enable it.")

if DO_BUILD_APP:
    run(
        f"{PYTHON} app/build_site.py --from-graph --graph {shlex.quote(SIMILARITY_GRAPH_INPUT)} --similarity-dir {shlex.quote(SIMILARITY_GRAPH_OUT)} --out {shlex.quote(APP_OUT)}"
    )
else:
    print("Skipping app export. Set DO_BUILD_APP = True to enable it.")


In [12]:
run(f"{PYTHON} pipeline/release_bundle.py")

$ /usr/local/bin/python3.11 pipeline/release_bundle.py
Created release bundle: /Users/talha/Code/free-exercise-graph/pipeline/releases/20260324-195340-release-bundle



CompletedProcess(args='/usr/local/bin/python3.11 pipeline/release_bundle.py', returncode=0, stdout='Created release bundle: /Users/talha/Code/free-exercise-graph/pipeline/releases/20260324-195340-release-bundle\n', stderr='')

## 9. Freeze a Release Bundle

This writes a timestamped bundle containing the current DB, graph, scorecard, and enrichment export if present.


In [ ]:
## 10. Start the MCP Server

Only run this when you want the graph available to MCP clients.
